# Métricas y normalización

## Problema real

Dos chunks parecidos pueden dar scores muy distintos si no normalizás

## Conceptos clave

Coseno, producto interno y distancia L2 pueden ordenar los mismos resultados de formas distintas. Si un chunk tiene un embedding "grande" (con mucha magnitud) pero apunta poco hacia la pregunta, el producto interno sin normalizar lo puede poner primero por error. Normalizar (que todos los vectores tengan longitud 1) evita esa trampa.

In [ ]:
# Dos embeddings de juguete en 2D para ver la "trampa de la magnitud".
import numpy as np

chunk_largo = np.array([10., 1.])   # vector "grande", pero apunta poco hacia la pregunta
chunk_corto = np.array([2., 2.])    # vector chico, pero apunta justo hacia la pregunta
pregunta = np.array([1., 1.])

for nombre, chunk in [("chunk largo", chunk_largo), ("chunk corto", chunk_corto)]:
    producto_interno = chunk @ pregunta
    coseno = producto_interno / (np.linalg.norm(chunk) * np.linalg.norm(pregunta))
    print(f"{nombre}: producto_interno={producto_interno:.1f}  coseno={coseno:.2f}")

## Qué pasó acá

Sin normalizar, el "chunk largo" gana (11.0 > 4.0) solo porque tiene más magnitud, aunque el "chunk corto" es el que realmente apunta en la misma dirección que la pregunta (coseno 1.00 vs 0.77). Por eso, si tu índice usa producto interno, normalizá los embeddings antes de guardarlos -o usá coseno directamente-: si no, tu chatbot va a priorizar chunks "largos" por sobre chunks relevantes.

## Errores comunes

Mezclar métricas: indexar con producto interno pero razonar como si fuera coseno (o al revés). Y usar embeddings de dos modelos distintos para chunks y pregunta: ni siquiera van a vivir en el mismo espacio.

## Viendo la diferencia de magnitud

El gráfico siguiente muestra los dos vectores sin normalizar: fijate que uno es mucho más largo que el otro, aunque el corto apunte más derecho hacia la pregunta.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

vectores = np.array([[10., 1.], [2., 2.], [1., 1.]])  # chunk largo, chunk corto, pregunta
etiquetas = ["chunk largo", "chunk corto", "pregunta"]
colores = ["#e76f51", "#2a9d8f", "#264653"]

plt.figure(figsize=(5, 4))
for vector, etiqueta, color in zip(vectores, etiquetas, colores):
    plt.quiver(0, 0, vector[0], vector[1], angles="xy", scale_units="xy", scale=1, color=color, label=etiqueta)
plt.xlim(0, 11)
plt.ylim(0, 4)
plt.legend()
plt.grid(alpha=.3)
plt.title("La magnitud no es lo mismo que la relevancia")
plt.show()